
# 🧪 Hybrid Text Features: **TF‑IDF ⊕ Doc2Vec** → XGBoost (BBC News)

This Colab‑ready notebook builds and compares three models on the BBC News dataset:

1. **TF‑IDF → XGBoost**  
2. **Doc2Vec → XGBoost**  
3. **Hybrid (TF‑IDF ⊕ Doc2Vec) → XGBoost**  ← concatenation of both feature sets

It reports **accuracy, precision, recall, F1 (weighted)**, and **confusion matrices** for each.


## 0) Setup

In [ ]:

# If running in Colab, uncomment to install deps
# !pip -q install gensim xgboost scikit-learn matplotlib tqdm

import os, re, string, multiprocessing
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
from tqdm import tqdm

# XGBoost
import xgboost as xgb

# Gensim Doc2Vec
from gensim.models.doc2vec import TaggedDocument, Doc2Vec

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1) Load BBC dataset

In [ ]:

# Option A: Use a local path if this file is already present (e.g., uploaded to Colab working dir)
CSV_PATHS = [
    '/mnt/data/bbc-text.csv',   # path used when running via ChatGPT workspace
    './bbc-text.csv'            # default Colab working directory path
]

csv_path = None
for p in CSV_PATHS:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    # Fallback: prompt user to upload if not found
    try:
        from google.colab import files  # type: ignore
        print("Upload 'bbc-text.csv' (two columns: category,text).")
        uploaded = files.upload()
        csv_path = list(uploaded.keys())[0]
    except Exception as e:
        raise FileNotFoundError("Could not find 'bbc-text.csv'. Please place it in the working directory or upload it.") from e

df = pd.read_csv(csv_path)
print(df.shape)
df.head()


## 2) Minimal Cleaning

In [ ]:

def clean_text(s: str) -> str:
    """Lightweight cleaner: lowercase, remove punctuation/numbers, collapse spaces."""
    s = s.lower()
    s = s.translate(str.maketrans('', '', string.punctuation))
    s = re.sub(r'\d+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


## 3) Doc2Vec transformer (sklearn-compatible)

In [ ]:

class Doc2VecTransformer:
    """Simple sklearn-like transformer for Gensim Doc2Vec."""
    def __init__(self, vector_size=200, epochs=20, learning_rate=0.02, min_alpha=0.0001, alpha_decay=0.95,
                 dm=1, window=5, min_count=2, workers=None, seed=42):
        self.vector_size = vector_size
        self.epochs = epochs
        self.learning_rate = float(learning_rate)
        self.min_alpha = float(min_alpha)
        self.alpha_decay = float(alpha_decay)
        self.dm = dm
        self.window = window
        self.min_count = min_count
        self.workers = (multiprocessing.cpu_count() - 1) if workers is None else int(workers)
        self.seed = seed
        self.model = None

    def _tagged(self, texts):
        return [TaggedDocument(clean_text(t).split(), [i]) for i, t in enumerate(texts)]

    def fit(self, texts, y=None):
        tagged = self._tagged(texts)
        model = Doc2Vec(
            vector_size=self.vector_size,
            dm=self.dm,
            window=self.window,
            min_count=self.min_count,
            workers=self.workers,
            seed=self.seed
        )
        model.build_vocab(tagged)

        alpha = self.learning_rate
        for _ in tqdm(range(self.epochs), desc="Doc2Vec training"):
            shuffled = tagged.copy()
            rng = np.random.RandomState(self.seed)
            rng.shuffle(shuffled)
            model.train(
                shuffled,
                total_examples=len(shuffled),
                epochs=1,
                start_alpha=alpha,
                end_alpha=alpha
            )
            alpha = max(self.min_alpha, alpha * self.alpha_decay)
            model.alpha = alpha
            model.min_alpha = alpha

        self.model = model
        return self

    def transform(self, texts):
        assert self.model is not None, "Call fit() before transform()."
        vecs = [self.model.infer_vector(clean_text(t).split(), alpha=self.min_alpha, steps=20, random_seed=self.seed) for t in texts]
        return np.asarray(vecs)

    def fit_transform(self, texts, y=None):
        return self.fit(texts, y).transform(texts)


## 4) Build feature sets: TF‑IDF, Doc2Vec, and Hybrid

In [ ]:

# Labels
y = df['category']

# TF-IDF
tfidf = TfidfVectorizer(stop_words='english', max_features=20000, ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(df['text'].map(clean_text))

# Doc2Vec (dense)
d2v = Doc2VecTransformer(vector_size=200, epochs=20)
X_d2v = d2v.fit_transform(df['text'])

# Hybrid stack
from scipy.sparse import csr_matrix, hstack
X_hybrid = hstack([X_tfidf, csr_matrix(X_d2v)], format='csr')

X_tfidf.shape, X_d2v.shape, X_hybrid.shape


## 5) Train/Test split

In [ ]:

Xtf_train, Xtf_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
Xdv_train, Xdv_test, _, _            = train_test_split(X_d2v,  y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
Xhy_train, Xhy_test, _, _            = train_test_split(X_hybrid,y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

Xtf_train.shape, Xdv_train.shape, Xhy_train.shape


## 6) Train XGBoost models

In [ ]:

def train_xgb(X_train, y_train, num_class):
    clf = xgb.XGBClassifier(
        n_estimators=600,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='multi:softprob',
        eval_metric='mlogloss',
        tree_method='hist',
        random_state=RANDOM_STATE
    )
    clf.fit(X_train, y_train)
    return clf

num_classes = y.nunique()

model_tfidf  = train_xgb(Xtf_train, y_train, num_classes)
model_d2v    = train_xgb(Xdv_train, y_train, num_classes)
model_hybrid = train_xgb(Xhy_train, y_train, num_classes)

"Training complete."


## 7) Evaluate (accuracy, precision, recall, F1, confusion matrix)

In [ ]:

def evaluate_model(model, X_test, y_test, title="Model"):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)

    print(f"\n=== {title} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {pr:.4f}  Recall: {rc:.4f}  F1: {f1:.4f}\n")
    print(classification_report(y_test, y_pred, zero_division=0))

    plt.figure(figsize=(6,5))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, xticks_rotation='vertical')
    plt.title(f"Confusion Matrix — {title}")
    plt.tight_layout()
    plt.show()

    return {"accuracy": acc, "precision_w": pr, "recall_w": rc, "f1_w": f1}

m_tfidf  = evaluate_model(model_tfidf,  Xtf_test, y_test, title="XGBoost (TF-IDF)")
m_d2v    = evaluate_model(model_d2v,    Xdv_test, y_test, title="XGBoost (Doc2Vec)")
m_hybrid = evaluate_model(model_hybrid, Xhy_test, y_test, title="XGBoost (Hybrid: TF-IDF ⊕ Doc2Vec)")


## 8) Compare models

In [ ]:

cmp = pd.DataFrame([
    {"model": "TF-IDF", **m_tfidf},
    {"model": "Doc2Vec", **m_d2v},
    {"model": "Hybrid", **m_hybrid},
]).set_index('model')
display(cmp)

# Bar chart for weighted F1
labels = cmp.index.tolist()
vals = cmp['f1_w'].values

plt.figure(figsize=(6,4))
plt.bar(labels, vals)
plt.ylim(0, 1.05)
plt.title("Weighted F1 by Model")
plt.tight_layout()
plt.show()
